# Step 10.1 — Encoder–Decoder Seq2Seq：从“读完整段输入”到“逐步生成输出”

Step 9.5 已经收束 recurrent family 的局限：LSTM/GRU 改善了长期信息与梯度流，但 recurrence、固定宽度 state 与时间串行依赖仍然存在。

历史上的下一步不是立刻发明 Transformer，而是先把 recurrent models 组织成一个新的任务结构：**Sequence-to-Sequence（Seq2Seq）Encoder–Decoder**。

本节目标只建立基本结构：

```text
source sequence -> encoder -> final encoder state
                              ↓
                         decoder initial state
                              ↓
                    target sequence generation
```

不分析这个 final-state bridge 的 bottleneck；那是 roadmap 的 Step 10.2。也不进入 Attention。


## 1. 新任务形式：Source Sequence 与 Target Sequence 可以有不同长度

之前 next-token language modeling 常在同一条 token stream 上做 shifted prediction。

Seq2Seq 处理的是：

```text
source sequence  ->  target sequence
输入序列             输出序列
```

例如机器翻译：

```text
source: I am here .
target: 我 在 这里 。
```

为了避免把两个序列长度都叫同一个 T，本节新增两个符号：

```text
T_src = source sequence length
T_tgt = target sequence length
```

它们不要求相等。

如果 source token IDs 的 shape 是：

$$
(B,T_{src})
$$

target token IDs 的 shape 可以是：

$$
(B,T_{tgt}).
$$

source 与 target 甚至可以使用不同 vocabulary：

```text
V_src = source vocabulary size
V_tgt = target vocabulary size
```

例如英译中时，两边 tokenizer/vocabulary 完全可以不同；同语言任务也可以共享 tokenizer。

### 两个新的 Special Tokens

自回归 decoder 需要知道“从哪里开始”和“什么时候结束”，所以常见：

```text
<BOS> = Beginning Of Sequence
<EOS> = End Of Sequence
```

例如 target：

```text
<BOS> 我 在 这里 。 <EOS>
```

`<BOS>/<EOS>` 都只是 vocabulary 中的特殊 token 类型，也有自己的 token ID。


## 2. Encoder 做什么？把整个 Source Sequence 读成一条 State Trajectory

先使用你已经熟悉的 GRU 作为 encoder。

source IDs：

$$
X_{src}\in\mathbb Z^{B\times T_{src}}.
$$

经过 source embedding：

$$
(B,T_{src})
\rightarrow
(B,T_{src},C_{src}).
$$

然后 encoder GRU：

$$
(B,T_{src},C_{src})
\rightarrow
H^{enc}\in\mathbb R^{B\times T_{src}\times H}.
$$

其中：

```text
H_enc[:, 0, :]           = source 第 0 个位置后的 encoder hidden state
H_enc[:, 1, :]           = source 第 1 个位置后的 encoder hidden state
...
H_enc[:, T_src-1, :]     = source 全部读完后的 final encoder state
```

定义：

$$
s
=
H^{enc}_{:,T_{src}-1,:}
\in
\mathbb R^{B\times H}.
$$

这里把 `s` 称为 **context state / encoder final state**。

Step 10.1 只把它当作 encoder 与 decoder 之间的桥梁；暂时不讨论“一个固定 H 维 state 是否足够承载长 source”的问题。


In [ ]:
import mlx.core as mx
import mlx.nn as nn

B = 2
T_src = 5
V_src = 20
C_src = 6
H = 8

src_embedding = nn.Embedding(num_embeddings=V_src, dims=C_src)
encoder = nn.GRU(input_size=C_src, hidden_size=H)

source_ids = mx.array([
    [3, 5, 7, 9, 2],
    [4, 6, 6, 8, 2],
])  # (B,T_src)

source_x = src_embedding(source_ids)       # (B,T_src,C_src)
encoder_states = encoder(source_x)         # (B,T_src,H)
context_state = encoder_states[:, -1, :]   # (B,H)

print('source_ids:', source_ids.shape)
print('source_x:', source_x.shape)
print('encoder_states:', encoder_states.shape)
print('context_state:', context_state.shape)


## 3. Decoder 做什么？把 Encoder Final State 当作初始 Hidden State

Decoder 也是 recurrent model，但它的职责不是继续读 source，而是**逐步生成 target sequence**。

最经典的 recurrent encoder–decoder bridge 是：

$$
h_{-1}^{dec}=s
$$

也就是把 encoder final state `s=(B,H)` 直接作为 decoder 的 initial hidden state。

decoder 每一步接收一个 target-side token embedding：

$$
y_t^{in}
\rightarrow
e_t^{tgt}\in\mathbb R^{B\times C_{tgt}}.
$$

然后：

$$
h_t^{dec}
=
GRU_{dec}(e_t^{tgt},h_{t-1}^{dec}).
$$

最后通过一个 output projection / LM head：

$$
logits_t
=
h_t^{dec}W_{out}^T+b_{out}
\in
\mathbb R^{B\times V_{tgt}}.
$$

这里 `logits` 仍然是对 target vocabulary 中每个 token 的未归一化分数；这个概念后面正式语言模型 loss 时还会再深入。

### 为什么 Encoder 和 Decoder 可以有不同参数？

它们承担不同角色：

```text
encoder: source -> source representation
decoder: previous target tokens + encoder context -> next target prediction
```

因此经典 Seq2Seq 通常有独立的 encoder / decoder recurrent parameters；是否共享 embedding 或其他权重取决于具体设计。


## 4. Decoder 在 Training 与 Inference 时吃什么输入？

这里需要一个新概念：**Teacher Forcing（教师强制）**。

假设 target token sequence 是：

```text
<BOS>  A  B  C  <EOS>
```

训练时常构造：

```text
decoder input : <BOS> A B C
target label  : A     B C <EOS>
```

也就是与 Step 6 的 next-token shift 很相似。

Teacher forcing 的意思是：训练 decoder 的第 t 步时，输入前一个**真实 target token**，而不是必须使用模型自己上一时刻预测出来的 token。

这样可以并行准备整条 target input tensor，也让训练信号更直接。

但 decoder recurrent state 依然沿 `T_tgt` 逐步更新，所以 recurrent dependency 仍然存在。

### Inference / Generation

真正生成时没有完整正确 target 可以提前提供。

因此流程是：

```text
<BOS>
  ↓
predict token_1
  ↓
把 token_1 作为下一步输入
  ↓
predict token_2
  ↓
...
  ↓
直到生成 <EOS> 或达到长度上限
```

所以 target output length `T_tgt` 可以和 source length `T_src` 完全不同。

Teacher forcing 是训练策略，不是 encoder–decoder 架构本身的必要数学组成。


In [ ]:
B = 2
T_tgt = 4
V_tgt = 30
C_tgt = 6

tgt_embedding = nn.Embedding(num_embeddings=V_tgt, dims=C_tgt)
decoder = nn.GRU(input_size=C_tgt, hidden_size=H)
output_proj = nn.Linear(H, V_tgt)

# Example shifted target-side inputs:
# each row starts with a hypothetical <BOS> token id 1
decoder_input_ids = mx.array([
    [1, 11, 12, 13],
    [1, 15, 16, 17],
])  # (B,T_tgt)

decoder_x = tgt_embedding(decoder_input_ids)  # (B,T_tgt,C_tgt)

# Encoder final state becomes decoder initial hidden state.
decoder_states = decoder(
    decoder_x,
    hidden=context_state,
)  # (B,T_tgt,H)

logits = output_proj(decoder_states)  # (B,T_tgt,V_tgt)

print('decoder_input_ids:', decoder_input_ids.shape)
print('decoder_x:', decoder_x.shape)
print('decoder_states:', decoder_states.shape)
print('logits:', logits.shape)


## 5. 把完整 Seq2Seq Data Flow 串起来

现在可以把 Step 10.1 压缩成一条完整 shape flow。

### Encoder

$$
(B,T_{src})
\xrightarrow{Embedding}
(B,T_{src},C_{src})
\xrightarrow{Encoder\ GRU}
(B,T_{src},H).
$$

取最后 encoder state：

$$
s=(B,H).
$$

### Bridge

$$
h_{-1}^{dec}=s.
$$

### Decoder

训练阶段 shifted target input：

$$
(B,T_{tgt})
\xrightarrow{Embedding}
(B,T_{tgt},C_{tgt})
\xrightarrow{Decoder\ GRU}
(B,T_{tgt},H)
\xrightarrow{Output\ Projection}
(B,T_{tgt},V_{tgt}).
$$

这个结构已经能够描述：

```text
长度为 T_src 的 source
↓
编码成 final state s
↓
从 s 开始 decoder
↓
产生长度为 T_tgt 的 target
```

所以 Seq2Seq 的核心突破不是“RNN 不再串行”，而是：

> **把一个输入序列的表示过程与另一个输出序列的生成过程拆成 Encoder 和 Decoder 两个角色，并用 recurrent state 把二者连接起来。**

### Variable-Length 的工程边界

概念上每个样本可以有不同 `T_src / T_tgt`。但同一个 dense Batch tensor 仍需要规则矩形，因此实际 batching 常需要 padding / masks / length bookkeeping。

这和前面 RNN 变长序列的工程边界一致；本节不展开 padding 实现。

### Step 10.1 材料边界

- source 与 target 是两个序列，`T_src` 与 `T_tgt` 可以不同。
- encoder 读取完整 source 并产生 hidden-state trajectory；经典 recurrent Seq2Seq 使用 final encoder state 作为 decoder initial state。
- decoder 根据 previous target-side token 与 recurrent state 逐步产生 target。
- `<BOS>/<EOS>` 是控制序列开始/结束的 special tokens。
- training 常使用 teacher forcing；inference 则把模型自己生成的前一 token 作为下一步输入。
- output projection 把 decoder hidden `(B,H)` 映射到 target vocabulary logits `(B,V_tgt)`。
- 本节只建立 encoder–decoder Seq2Seq 基本结构；不分析 fixed-vector bottleneck，不进入 Step 10.2。
